# Aggregate Report from Single-Model Notebook Outputs

This notebook collects saved outputs from the **six single-model training notebooks** and builds:

1. **Results Comparison Table**
2. **Learning Curves Comparison**
3. **SHAP Feature Comparison**
4. **Visual Performance Visualizations**
5. **Comparison vs paper targets**

## Accepted inputs
This patched notebook supports either of these workflows:

### A. Upload result zip files
Upload the zip files created at the end of each single-model notebook. The notebook will reconstruct the expected `results/...` folder structure automatically.

### B. Existing extracted runs
If `results/...` already contains extracted runs, the notebook will use them directly.

## Supported run layouts
It can read both of these save formats:

### Legacy single-model layout
```text
results/<safe_model>_run_<timestamp>/
  tables/model_results.csv
  tables/all_results.json
  tables/shap_importance.csv
  training_history/*_history.json
```

### Patched flat run layout
```text
results/<safe_model>/<timestamp>/
  metrics.json
  history.csv
  top_shap_features_df.csv
  classification_report.txt
  artifacts/
  figures/
```


In [ ]:
# CELL 1: Imports
import os
import re
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')


In [ ]:
# CELL 1A: Upload zip files and reconstruct the expected results/ structure

import io
import os
import re
import shutil
import zipfile
from pathlib import Path

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

RESULTS_ROOT = Path("results")
RESULTS_ROOT.mkdir(exist_ok=True)

MODEL_MAP = {
    "lstm_cnn_hybrid": "lstm_cnn_hybrid",
    "cnn": "cnn",
    "rnn": "rnn",
    "lstm": "lstm",
    "bilstm": "bilstm",
    "gru": "gru",
}

ZIP_PATTERN = re.compile(r"^(.*)_(\d{8}_\d{6})\.zip$", re.IGNORECASE)


def parse_zip_name(fname: str):
    m = ZIP_PATTERN.match(os.path.basename(fname))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def extract_run_zip_bytes(fname: str, data: bytes):
    slug_raw, timestamp = parse_zip_name(fname)
    if slug_raw is None or timestamp is None:
        print(f"Skipping unrecognized zip filename: {fname}")
        return
    if slug_raw not in MODEL_MAP:
        print(f"Skipping unknown model slug in zip name: {slug_raw}")
        return

    model_slug = MODEL_MAP[slug_raw]
    target_dir = RESULTS_ROOT / f"{model_slug}_run_{timestamp}"
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {fname} -> {target_dir}")
    with zipfile.ZipFile(io.BytesIO(data), "r") as zf:
        zf.extractall(target_dir)

print("If you already extracted runs into results/, you can skip upload.")
if IN_COLAB:
    print("Upload the zip files downloaded from each single-model notebook.")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        if not fname.lower().endswith(".zip"):
            print(f"Skipping non-zip file: {fname}")
            continue
        extract_run_zip_bytes(fname, data)
    print("\nUpload + extraction complete.")
else:
    print("Not running inside Colab. Place zip files beside the notebook and run a local extraction cell if needed.")


In [ ]:
# CELL 1B: Check what run folders are available

from pathlib import Path

RESULTS_ROOT = Path("results")
if not RESULTS_ROOT.exists():
    print("results/ does not exist yet.")
else:
    for root, dirs, files in os.walk(RESULTS_ROOT):
        level = str(root).replace(str(RESULTS_ROOT), "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(files)[:8]:
            print(f"{indent}  {f}")


In [ ]:
# CELL 2: Discover the latest run for each model (robust)
from pathlib import Path

RESULTS_ROOT = Path("results")

MODEL_PATTERNS = {
    "LSTM-CNN (Hybrid)": ["lstm_cnn_hybrid_run_*", "lstm_cnn_hybrid/*"],
    "CNN": ["cnn_run_*", "cnn/*"],
    "RNN": ["rnn_run_*", "rnn/*"],
    "LSTM": ["lstm_run_*", "lstm/*"],
    "BiLSTM": ["bilstm_run_*", "bilstm/*"],
    "GRU": ["gru_run_*", "gru/*"],
}


def latest_dir(patterns):
    matches = []
    for pattern in patterns:
        matches.extend([p for p in RESULTS_ROOT.glob(pattern) if p.is_dir()])
    matches = sorted(matches)
    return matches[-1] if matches else None

latest_runs = {model: latest_dir(patterns) for model, patterns in MODEL_PATTERNS.items()}

print("Latest runs discovered:")
for model, path in latest_runs.items():
    print(f"  {model:<18} -> {path}")

missing = [m for m, p in latest_runs.items() if p is None]
if missing:
    print("\nWarning: missing saved runs for:", ", ".join(missing))
    print("The notebook will continue with available models.")


In [ ]:
# CELL 3: Load artifacts from each run (supports legacy + patched layouts)
import json
import re
from pathlib import Path

results_rows = []
histories = {}
shap_tables = {}
run_meta = {}
loaded_models = []


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_history(run_dir: Path):
    history_jsons = list((run_dir / "training_history").glob("*_history.json"))
    if history_jsons:
        hist = read_json(history_jsons[0])
        return {k: list(v) for k, v in hist.items()}

    history_csv = run_dir / "history.csv"
    if history_csv.exists():
        hdf = pd.read_csv(history_csv)
        hist = {}
        for col in hdf.columns:
            if col.lower() == "epoch":
                continue
            hist[col] = hdf[col].tolist()
        return hist

    return None


def load_shap(run_dir: Path):
    shap_csv = run_dir / "tables" / "shap_importance.csv"
    if shap_csv.exists():
        sdf = pd.read_csv(shap_csv)
        if "Feature" in sdf.columns and "SHAP_importance" in sdf.columns:
            return sdf[["Feature", "SHAP_importance"]]

    for candidate in [run_dir / "top_shap_features_df.csv", run_dir / "shap_feature_importance_df.csv"]:
        if candidate.exists():
            sdf = pd.read_csv(candidate)
            rename_map = {}
            if "feature" in sdf.columns:
                rename_map["feature"] = "Feature"
            if "mean_abs_shap" in sdf.columns:
                rename_map["mean_abs_shap"] = "SHAP_importance"
            sdf = sdf.rename(columns=rename_map)
            if "Feature" in sdf.columns and "SHAP_importance" in sdf.columns:
                return sdf[["Feature", "SHAP_importance"]]

    return None


def load_metrics_row(model: str, run_dir: Path):
    model_results_path = run_dir / "tables" / "model_results.csv"
    if model_results_path.exists():
        mr = pd.read_csv(model_results_path, index_col=0)
        if len(mr) == 1:
            row = mr.iloc[0].to_dict()
        elif model in mr.index:
            row = mr.loc[model].to_dict()
        else:
            row = mr.iloc[0].to_dict()
        row["Model"] = model
        return row

    all_results_path = run_dir / "tables" / "all_results.json"
    if all_results_path.exists():
        data = read_json(all_results_path)
        metrics = data.get("metrics", {})
        if model in metrics:
            row = dict(metrics[model])
        elif metrics:
            row = dict(next(iter(metrics.values())))
        else:
            row = {}
        row["Model"] = model
        return row

    metrics_json = run_dir / "metrics.json"
    if metrics_json.exists():
        row = read_json(metrics_json)
        row["Model"] = model
        return row

    return None


def extract_run_meta(model: str, run_dir: Path, row: dict):
    all_results_path = run_dir / "tables" / "all_results.json"
    if all_results_path.exists():
        data = read_json(all_results_path)
        rcfg = data.get("run_config", {})
        if "train_time_sec" in rcfg and "Train Time (s)" not in row:
            row["Train Time (s)"] = rcfg["train_time_sec"]
        if "epochs_ran" in rcfg and "Epochs Run" not in row:
            row["Epochs Run"] = rcfg["epochs_ran"]
        run_meta[model] = rcfg
        return

    meta_json = run_dir / "metadata.json"
    if meta_json.exists():
        run_meta[model] = read_json(meta_json)


for model, run_dir in latest_runs.items():
    if run_dir is None:
        continue

    run_dir = Path(run_dir)
    row = load_metrics_row(model, run_dir)
    if row is None:
        print(f"Skipping {model}: no metrics found in {run_dir}")
        continue

    hist = load_history(run_dir)
    sdf = load_shap(run_dir)
    extract_run_meta(model, run_dir, row)

    if hist is not None and "Epochs Run" not in row:
        ref_key = "val_accuracy" if "val_accuracy" in hist else next(iter(hist.keys()), None)
        if ref_key is not None:
            row["Epochs Run"] = len(hist[ref_key])

    results_rows.append(row)
    if hist is not None:
        histories[model] = hist
    if sdf is not None:
        shap_tables[model] = sdf
    loaded_models.append(model)

results_df = pd.DataFrame(results_rows).set_index("Model") if results_rows else pd.DataFrame()

for col in [
    'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
    'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run'
]:
    if col in results_df.columns:
        results_df[col] = pd.to_numeric(results_df[col], errors='coerce')

if results_df.empty:
    raise FileNotFoundError(
        "No valid saved runs could be loaded from results/.\n"
        "Upload/extract the single-model zip files first."
    )

print("Loaded models:", ", ".join(loaded_models))
display(results_df)


In [ ]:
# CELL 4: Add paper targets for comparison (conservative / citation-safe)
import numpy as np
import pandas as pd

# Use only targets explicitly stated in the paper text / tables.
# Available from the paper:
# - Hybrid: accuracy, precision, recall, F1, FPR, robustness
# - CNN/LSTM/GRU/CNN-LSTM Hybrid: accuracy, FPR, robustness
# - BiLSTM and Simple RNN: accuracy only is stated in the discussion text
paper_targets = pd.DataFrame(index=results_df.index)

if 'Accuracy (%)' in results_df.columns:
    paper_targets['Paper Accuracy (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 99.87,
        'BiLSTM': 98.92,
        'LSTM': 98.34,
        'GRU': 97.89,
        'CNN': 97.45,
        'RNN': 95.78,
    })

if 'Precision (%)' in results_df.columns:
    paper_targets['Paper Precision (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 99.89,
    })

if 'Recall (%)' in results_df.columns:
    paper_targets['Paper Recall (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 99.85,
    })

if 'F1-Score (%)' in results_df.columns:
    paper_targets['Paper F1-Score (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 99.87,
    })

if 'FPR (%)' in results_df.columns:
    paper_targets['Paper FPR (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 0.13,
        'LSTM': 1.58,
        'GRU': 2.05,
        'CNN': 2.38,
        # Table 7 uses CNN-LSTM hybrid rather than BiLSTM / RNN.
        # We leave unsupported entries blank rather than invent values.
    })

if 'Adversarial Robustness (%)' in results_df.columns:
    paper_targets['Paper Adversarial Robustness (%)'] = pd.Series({
        'LSTM-CNN (Hybrid)': 90.2,
        'LSTM': 85.5,
        'GRU': 84.0,
        'CNN': 82.0,
    })

comparison_df = results_df.copy()

for metric_col in results_df.columns:
    paper_col = f'Paper {metric_col}'
    if paper_col in paper_targets.columns:
        comparison_df[paper_col] = paper_targets[paper_col]
        if 'FPR' in metric_col:
            comparison_df[f'Delta {metric_col}'] = comparison_df[metric_col] - comparison_df[paper_col]
        else:
            comparison_df[f'Delta {metric_col}'] = comparison_df[metric_col] - comparison_df[paper_col]

available_target_counts = paper_targets.notna().sum().to_dict()
print("Paper target coverage by metric:")
for k, v in available_target_counts.items():
    print(f"  {k:<35} {int(v)} model(s)")

comparison_df


In [ ]:
# CELL 5: Save comparison tables for report use
REPORT_DIR = Path("aggregated_report")
(REPORT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)

results_df.to_csv(REPORT_DIR / "tables" / "results_comparison_table.csv")
comparison_df.to_csv(REPORT_DIR / "tables" / "results_vs_paper_table.csv")
paper_targets.to_csv(REPORT_DIR / "tables" / "paper_target_table.csv")

print("Saved:")
print(" ", REPORT_DIR / "tables" / "results_comparison_table.csv")
print(" ", REPORT_DIR / "tables" / "results_vs_paper_table.csv")
print(" ", REPORT_DIR / "tables" / "paper_target_table.csv")


# CELL 6: Learning Curves Comparison (validation accuracy + validation loss)
if not histories:
    print("No training histories available to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for model, hist in histories.items():
        if 'val_accuracy' in hist:
            axes[0].plot(hist['val_accuracy'], label=model, linewidth=2)
        if 'val_loss' in hist:
            axes[1].plot(hist['val_loss'], label=model, linewidth=2)

    axes[0].set_title("Validation Accuracy by Model", fontweight='bold')
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Validation Accuracy")

    axes[1].set_title("Validation Loss by Model", fontweight='bold')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation Loss")

    for ax in axes:
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(REPORT_DIR / "figures" / "learning_curves_comparison.png", dpi=220, bbox_inches='tight')
    plt.show()


In [ ]:
# CELL 7: Best epoch summary
best_epoch_rows = []
for model, hist in histories.items():
    if 'val_accuracy' not in hist or 'val_loss' not in hist:
        continue
    val_acc = np.array(hist['val_accuracy'])
    val_loss = np.array(hist['val_loss'])
    best_epoch_rows.append({
        "Model": model,
        "Best Val Accuracy Epoch": int(np.argmax(val_acc) + 1),
        "Best Val Accuracy": float(np.max(val_acc)),
        "Best Val Loss Epoch": int(np.argmin(val_loss) + 1),
        "Best Val Loss": float(np.min(val_loss)),
        "Epochs Run": len(val_acc),
    })

best_epochs_df = pd.DataFrame(best_epoch_rows).set_index("Model") if best_epoch_rows else pd.DataFrame()
if best_epochs_df.empty:
    print("No history files available for best-epoch summary.")
else:
    best_epochs_df.to_csv(REPORT_DIR / "tables" / "best_epochs_summary.csv")
    display(best_epochs_df)


In [ ]:
# CELL 7: Best epoch summary
best_epoch_rows = []
for model, hist in histories.items():
    val_acc = np.array(hist['val_accuracy'])
    val_loss = np.array(hist['val_loss'])
    best_epoch_rows.append({
        "Model": model,
        "Best Val Accuracy Epoch": int(np.argmax(val_acc) + 1),
        "Best Val Accuracy": float(np.max(val_acc)),
        "Best Val Loss Epoch": int(np.argmin(val_loss) + 1),
        "Best Val Loss": float(np.min(val_loss)),
        "Epochs Run": len(val_acc),
    })

best_epochs_df = pd.DataFrame(best_epoch_rows).set_index("Model").sort_index()
best_epochs_df.to_csv(REPORT_DIR / "tables" / "best_epochs_summary.csv")
best_epochs_df


# CELL 8: Top SHAP features per model (small multiples)
TOP_N = 15

if not shap_tables:
    print("No SHAP tables available to plot.")
else:
    n_models = len(shap_tables)
    fig, axes = plt.subplots(n_models, 1, figsize=(10, 4 * n_models))
    if n_models == 1:
        axes = [axes]

    for ax, (model, sdf) in zip(axes, sorted(shap_tables.items())):
        top = sdf.sort_values("SHAP_importance", ascending=False).head(TOP_N).iloc[::-1]
        ax.barh(top["Feature"], top["SHAP_importance"], color="#2196F3", alpha=0.85)
        ax.set_title(f"Top {TOP_N} SHAP Features — {model}", fontweight='bold')
        ax.set_xlabel("SHAP importance")
        ax.set_ylabel("")

    plt.tight_layout()
    plt.savefig(REPORT_DIR / "figures" / "shap_top_features_by_model.png", dpi=220, bbox_inches='tight')
    plt.show()


In [ ]:
# CELL 9: SHAP heatmap across models (union of top features)
TOP_PER_MODEL = 20

if not shap_tables:
    print("No SHAP tables available for heatmap.")
else:
    feature_union = set()
    for model, sdf in shap_tables.items():
        feature_union.update(sdf.sort_values("SHAP_importance", ascending=False).head(TOP_PER_MODEL)["Feature"].tolist())

    def sort_feature_key(x):
        m = re.findall(r'\d+', str(x))
        return int(m[0]) if m else str(x)

    feature_union = sorted(feature_union, key=sort_feature_key)

    heatmap_df = pd.DataFrame(index=sorted(shap_tables.keys()), columns=feature_union, dtype=float)

    for model, sdf in shap_tables.items():
        tmp = sdf.set_index("Feature")["SHAP_importance"]
        for feat in feature_union:
            if feat in tmp.index:
                heatmap_df.loc[model, feat] = tmp.loc[feat]

    heatmap_df = heatmap_df.fillna(0.0)
    heatmap_df.to_csv(REPORT_DIR / "tables" / "shap_heatmap_values.csv")

    plt.figure(figsize=(max(12, len(feature_union) * 0.35), 6))
    sns.heatmap(heatmap_df, cmap="Blues", cbar_kws={"label": "SHAP importance"})
    plt.title("SHAP Feature Importance Across Models", fontweight='bold')
    plt.xlabel("Feature")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "figures" / "shap_heatmap_across_models.png", dpi=220, bbox_inches='tight')
    plt.show()


In [ ]:
# CELL 9: SHAP heatmap across models (union of top features)
TOP_PER_MODEL = 20

feature_union = set()
for model, sdf in shap_tables.items():
    feature_union.update(sdf.sort_values("SHAP_importance", ascending=False).head(TOP_PER_MODEL)["Feature"].tolist())

feature_union = sorted(feature_union, key=lambda x: int(re.findall(r'\d+', x)[0]) if re.findall(r'\d+', x) else x)

heatmap_df = pd.DataFrame(index=sorted(shap_tables.keys()), columns=feature_union, dtype=float)

for model, sdf in shap_tables.items():
    s = sdf.set_index("Feature")["SHAP_importance"]
    for feat in feature_union:
        heatmap_df.loc[model, feat] = float(s.get(feat, 0.0))

# keep strongest columns
top_cols = heatmap_df.mean(axis=0).sort_values(ascending=False).head(25).index
heatmap_df = heatmap_df[top_cols]

plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_df, cmap="YlGnBu", annot=False)
plt.title("SHAP Importance Heatmap Across Models", fontweight='bold')
plt.xlabel("Feature")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "shap_heatmap_across_models.png", bbox_inches='tight')
plt.show()

heatmap_df.to_csv(REPORT_DIR / "tables" / "shap_heatmap_values.csv")


# CELL 10: Results Comparison Table (styled)
desired_display_cols = [
    'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
    'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run'
]
display_cols = [c for c in desired_display_cols if c in results_df.columns]

styled = (results_df[display_cols]
          .sort_values('Accuracy (%)', ascending=False)
          .style
          .format('{:.2f}')
          .background_gradient(subset=[c for c in ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)','AUC-ROC (%)'] if c in display_cols], cmap='Greens')
          .background_gradient(subset=[c for c in ['FPR (%)','Train Time (s)'] if c in display_cols], cmap='Reds_r'))

display(styled)


In [ ]:
# CELL 11: Metric bar charts
metric_order = [m for m in ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'FPR (%)', 'Train Time (s)'] if m in results_df.columns]
if not metric_order:
    print("No comparable numeric metrics available for bar charts.")
else:
    n = len(metric_order)
    rows = int(np.ceil(n / 3))
    fig, axes = plt.subplots(rows, 3, figsize=(16, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    plot_df = results_df.sort_values('Accuracy (%)', ascending=False) if 'Accuracy (%)' in results_df.columns else results_df.copy()

    for ax, metric in zip(axes, metric_order):
        sns.barplot(x=plot_df.index, y=plot_df[metric], ax=ax, palette='viridis')
        ax.set_title(metric, fontweight='bold')
        ax.set_xlabel("")
        ax.tick_params(axis='x', rotation=30)

    for ax in axes[len(metric_order):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(REPORT_DIR / "figures" / "metric_bar_charts.png", dpi=220, bbox_inches='tight')
    plt.show()


# CELL 10A: Comparison Against Paper Targets

In [ ]:
# CELL 10A: Styled results vs paper targets
vs_paper_display_cols = []

for col in [
    'Accuracy (%)', 'Paper Accuracy (%)', 'Delta Accuracy (%)',
    'Precision (%)', 'Paper Precision (%)', 'Delta Precision (%)',
    'Recall (%)', 'Paper Recall (%)', 'Delta Recall (%)',
    'F1-Score (%)', 'Paper F1-Score (%)', 'Delta F1-Score (%)',
    'FPR (%)', 'Paper FPR (%)', 'Delta FPR (%)'
]:
    if col in comparison_df.columns:
        vs_paper_display_cols.append(col)

styled_vs_paper = (
    comparison_df[vs_paper_display_cols]
    .sort_values('Accuracy (%)', ascending=False)
    .style
    .format('{:.2f}', na_rep='—')
)

good_delta_cols = [c for c in comparison_df.columns if c.startswith('Delta ') and 'FPR' not in c and c in vs_paper_display_cols]
bad_delta_cols  = [c for c in comparison_df.columns if c.startswith('Delta ') and 'FPR' in c and c in vs_paper_display_cols]

if good_delta_cols:
    styled_vs_paper = styled_vs_paper.background_gradient(subset=good_delta_cols, cmap='Greens')
if bad_delta_cols:
    styled_vs_paper = styled_vs_paper.background_gradient(subset=bad_delta_cols, cmap='Reds_r')

display(styled_vs_paper)


In [ ]:
# CELL 11: Metric bar charts
metric_order = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'FPR (%)', 'Train Time (s)']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

plot_df = results_df.sort_values('Accuracy (%)', ascending=False)

for ax, metric in zip(axes, metric_order):
    sns.barplot(x=plot_df.index, y=plot_df[metric], ax=ax, palette='viridis')
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=30)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', fontsize=8, padding=2)

plt.suptitle("Metric Comparison Across Models", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "metric_bar_charts.png", bbox_inches='tight')
plt.show()


In [ ]:
# CELL 13: Compact report summary
summary_data = {}
for model in results_df.index:
    row = {}
    if model in histories and 'val_accuracy' in histories[model]:
        row['Best Validation Accuracy'] = max(histories[model]['val_accuracy'])
    if model in histories and 'val_loss' in histories[model]:
        row['Best Validation Loss'] = min(histories[model]['val_loss'])
    if 'Accuracy (%)' in results_df.columns:
        row['Final Test Accuracy (%)'] = results_df.loc[model, 'Accuracy (%)']
    if 'F1-Score (%)' in results_df.columns:
        row['Final Test F1 (%)'] = results_df.loc[model, 'F1-Score (%)']
    if 'Train Time (s)' in results_df.columns:
        row['Train Time (s)'] = results_df.loc[model, 'Train Time (s)']
    if 'Paper Accuracy (%)' in comparison_df.columns:
        row['Paper Target Accuracy (%)'] = comparison_df.loc[model, 'Paper Accuracy (%)']
    if 'Delta Accuracy (%)' in comparison_df.columns:
        row['Gap vs Paper Accuracy (%)'] = comparison_df.loc[model, 'Delta Accuracy (%)']
    if 'Paper FPR (%)' in comparison_df.columns:
        row['Paper Target FPR (%)'] = comparison_df.loc[model, 'Paper FPR (%)']
    if 'Delta FPR (%)' in comparison_df.columns:
        row['Gap vs Paper FPR (%)'] = comparison_df.loc[model, 'Delta FPR (%)']
    summary_data[model] = row

summary = pd.DataFrame(summary_data).T
if not summary.empty and 'Final Test Accuracy (%)' in summary.columns:
    summary = summary.sort_values('Final Test Accuracy (%)', ascending=False)
summary.to_csv(REPORT_DIR / "tables" / "report_summary.csv")
display(summary)


In [ ]:
# CELL 13: Compact report summary
summary = pd.DataFrame({
    'Best Validation Accuracy': {m: max(histories[m]['val_accuracy']) for m in histories},
    'Best Validation Loss':     {m: min(histories[m]['val_loss']) for m in histories},
    'Final Test Accuracy (%)':  results_df['Accuracy (%)'],
    'Final Test F1 (%)':        results_df['F1-Score (%)'],
    'Train Time (s)':           results_df['Train Time (s)'],
}).sort_values('Final Test Accuracy (%)', ascending=False)

summary.to_csv(REPORT_DIR / "tables" / "report_summary.csv")
summary


## How to use these outputs in your report

- **SHAP Feature section**  
  Use:
  - `aggregated_report/figures/shap_top_features_by_model.png`
  - `aggregated_report/figures/shap_heatmap_across_models.png`
  - `aggregated_report/tables/shap_heatmap_values.csv`

- **Learning Curves Comparison**  
  Use:
  - `aggregated_report/figures/learning_curves_comparison.png`
  - `aggregated_report/tables/best_epochs_summary.csv`

- **Visualizations**  
  Use:
  - `aggregated_report/figures/metric_bar_charts.png`
  - `aggregated_report/figures/radar_core_metrics.png`

- **Results Comparison Table**  
  Use:
  - `aggregated_report/tables/results_comparison_table.csv`
  - `aggregated_report/tables/results_vs_paper_table.csv`
  - `aggregated_report/tables/report_summary.csv`

If you want, you can also zip the `aggregated_report/` folder after running this notebook.


## Note on paper-target comparison

The paper provides **complete headline targets** for the proposed Hybrid LSTM-CNN, but only **partial target metrics** for some baselines in the accessible text and tables. This notebook therefore compares against paper targets **only where the paper explicitly reports them**, and leaves unsupported entries blank instead of inferring values.
